# Pan-cancer index / ontology inspect

Does **not** pool UNI embeddings or train. Use the CLI for GPU training (`demo.sh`).

```bash
conda activate SeededNTM
python -u code/PanCancer/prepare_pancancer_index.py
python -u code/PanCancer/PanCancer_train_validate_cv_UNIlabel.py \
  --use-spatial-context --spatial-k 8 --steps prepare train he_validate
```


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
PAN = REPO / "code" / "PanCancer"
DATA = REPO / "data" / "PanCancer"
for p in (PAN, REPO / "code" / "Hist2Pheno_pkg"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from pancancer_cv import fold_plan_frame
from pancancer_labels import load_pancancer_ontology, native_to_pancancer, ontology_class_names, remap_obs_frame
from pancancer_paths import COHORT_ORDER, discover_h5ads

heads = ontology_class_names()
onto = load_pancancer_ontology()
print("L1", heads["L1"])
print("L12", heads["L12"])
print("L2", heads["L2"])
display(onto[["L1", "L12", "L2", "present_in", "trainable"]])

L1 ['Epithelial', 'Immune', 'Stromal', 'Endothelial', 'Neural']
L12 ['Tumor', 'Epithelial', 'T_cell', 'B_Plasma', 'Myeloid', 'Stromal', 'Endothelial', 'Neural']
L2 ['DCIS', 'Tumor', 'Alveolar', 'Epithelial', 'Injury_epithelial', 'Myoepithelial', 'CD4_T', 'CD8_T', 'Tregs', 'NK', 'B_cell', 'Plasma', 'DC', 'Macrophage', 'Neutrophil', 'Fibroblast', 'Myofibroblast', 'Stromal', 'Endothelial', 'Neural']


,L1,L12,L2,present_in,trainable
0,Epithelial,Tumor,DCIS,BRCA,Y
1,Epithelial,Tumor,Tumor,BRCA; HCC; GBM; CRC,Y
2,Epithelial,Epithelial,Alveolar,Lung,Y
3,Epithelial,Epithelial,Epithelial,Lung; PDAC; GIST; CRC,Y
4,Epithelial,Epithelial,Injury_epithelial,Lung,Y
5,Epithelial,Epithelial,Myoepithelial,BRCA,Y
6,Immune,T_cell,CD4_T,Lung; BRCA; HCC; PDAC; GIST; CRC,Y
7,Immune,T_cell,CD8_T,Lung; BRCA; HCC; PDAC; CRC,Y
8,Immune,T_cell,Tregs,Lung; PDAC; GIST,Y
9,Immune,T_cell,NK,Lung; CRC,Y


In [2]:
rows = discover_h5ads()
disc = pd.DataFrame(rows)
print("discovered h5ads", len(disc))
print(disc.groupby("dataset").size().reindex(COHORT_ORDER))

idx_path = DATA / "section_index.csv"
if idx_path.is_file():
    idx = pd.read_csv(idx_path)
    print("\nsection_index", idx_path, "rows", len(idx))
    print(idx.groupby("dataset")[["n_keep", "n_raw"]].sum().reindex(COHORT_ORDER))
    plan = fold_plan_frame(idx, n_splits=5, seed=42)
    print("\nval sections per fold × dataset")
    display(pd.crosstab(plan["fold"], plan["dataset"]).reindex(columns=COHORT_ORDER).fillna(0).astype(int))
else:
    print("Run prepare_pancancer_index.py to write", idx_path)

discovered h5ads 896
dataset
Lung     25
BRCA      2
CRC       3
HCC      36
PDAC    278
GIST    550
GBM       2
dtype: int64

section_index /home/lingyu/ssd2/Python/Hist2Pheno/data/PanCancer/section_index.csv rows 896
          n_keep    n_raw
dataset                  
Lung      578077   637727
BRCA      264196   264495
CRC       468513   494004
HCC      1059094  1092742
PDAC     1393006  1393006
GIST     1693169  1784569
GBM       171016   175210

val sections per fold × dataset


dataset,Lung,BRCA,CRC,HCC,PDAC,GIST,GBM
fold,,,,,,,
0,5,1,1,8,56,110,1
1,5,1,1,7,56,110,1
2,5,0,1,7,56,110,0
3,5,0,0,7,55,110,0
4,5,0,0,7,55,110,0


In [3]:
import anndata as ad

idx_path = DATA / "section_index.csv"
if not idx_path.is_file():
    raise FileNotFoundError(idx_path)
idx = pd.read_csv(idx_path)
ok = idx[idx["ok"].astype(str).str.lower().isin(["true", "1"])].copy()
row = ok.iloc[0]
print("example", row["section_id"])
adata = ad.read_h5ad(row["h5ad"], backed="r")
obs = remap_obs_frame(adata.obs.copy(), native_col=row["native_col"])
print("raw", adata.n_obs, "kept", len(obs))
print(obs["final_CT"].value_counts().head(20))
print(obs["final_sublineage"].value_counts())
print(obs["final_lineage"].value_counts())
try:
    adata.file.close()
except Exception:
    pass

example Lung::THD0008
raw 36781 kept 34402
final_CT
Endothelial          9271
Macrophage           6947
Alveolar             6262
Fibroblast           3657
Neutrophil           3194
NK                   1478
Epithelial           1392
Myofibroblast        1199
CD4_T                 371
CD8_T                 324
Injury_epithelial     133
DC                     89
Plasma                 60
B_cell                 17
Tregs                   8
Name: count, dtype: int64
final_sublineage
Myeloid        10230
Endothelial     9271
Epithelial      7787
Stromal         4856
T_cell          2181
B_Plasma          77
Name: count, dtype: int64
final_lineage
Immune         12488
Endothelial     9271
Epithelial      7787
Stromal         4856
Name: count, dtype: int64
